# NiyamTrace-X Experiment 17 — 32-Request Native Source **Pilot**

This is an update to the **exact notebook you uploaded**, using your existing 32 frozen requests and the v3 corrected annotation CSVs. The extra experiment runs the **real NiyamTrace-X source pipeline** at pinned SHA `d3df6130ab7615e3f3692828b8193e5ad62b8af7` with `LLM_BACKEND=mock` (deterministic offline parser; **NOT** a live LLM), against **fresh synthetic SQLite per request**.

**Arms:** (1) `compiler_only` is a **proposed action** and **does not execute a write**, (2) `niyamtrace` runs full source `NiyamPipeline`, gate and broker. Treat the comparison as a diagnostic proposal-vs-gate benchmark, not a real ungated native-effect comparison.

**Scientific limitations:** 32 human-elicited, consent-indicated requests from 4 respondents; pilot/test split 16/16; annotations are provisionally labeled under an unapproved policy (28 BLOCK, 4 CLARIFY, 0 ALLOW/ESCALATE), so no final security or utility conclusion. `effect_audited=false` throughout; zero values in `unauthorized_effect_rows` mean *not assessed*, never verified zero risk. Nothing is uploaded to GitHub.

**Run on a Colab CPU** with internet access for repository checkout. The notebook fails rather than inventing predictions or continuing on missing files.
**Extraction fix:** the upload cell now selects the unique known ZIP root and removes old Colab copies before extraction.


## 1. Upload the complete study kit ZIP

In [ ]:
from google.colab import files
from pathlib import Path
from zipfile import ZipFile
import shutil
import sys

uploaded = files.upload()
zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
if len(zip_names) != 1:
    raise ValueError("Upload exactly one ZIP: NTX17_Today_Pilot_Kit.zip")

# Extract cleanly: previous Colab attempts may have left duplicate package roots.
PACKAGE_PARENT = Path("/content/NTX17_today")
if PACKAGE_PARENT.exists():
    shutil.rmtree(PACKAGE_PARENT)
PACKAGE_PARENT.mkdir(parents=True, exist_ok=True)

with ZipFile(zip_names[0]) as z:
    for member in z.infolist():
        path = Path(member.filename)
        if path.is_absolute() or ".." in path.parts:
            raise ValueError("Unsafe ZIP entry: " + member.filename)
    z.extractall(PACKAGE_PARENT)

# Known package structure inside the supplied ZIP: select it directly.
ROOT = PACKAGE_PARENT / "NTX17_Today_Pilot_Kit"
if not (ROOT / "core" / "exp17_gold.py").is_file():
    raise FileNotFoundError(f"Missing experiment core: {ROOT / 'core' / 'exp17_gold.py'}")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("Extraction successful")
print("Experiment root:", ROOT)
print("Pilot reviewer CSV files:", len(list((ROOT / "pilot_annotators").glob("*.csv"))))
print("Ready for section 2")


## 2. Prepare **unchanged** frozen human requests and pilot split

In [ ]:
from core.exp17_gold import prepare, adjudicate, evaluate
from core.exp17_native_export import export_native_predictions
import pandas as pd
import json, shutil

RAW = ROOT/'inputs'/'raw_natural_cases.csv'
OUT = ROOT/'outputs'/'experiment17'
OUT.mkdir(parents=True, exist_ok=True)
FROZEN = OUT/'frozen_cases_PRIVATE.csv'
PRED = ROOT/'inputs'/'actual_niyamtrace_predictions.csv'
A=OUT/'annotator_A_FILLED.csv';B=OUT/'annotator_B_FILLED.csv'
ADJ=OUT/'adjudicator_FILLED.csv'

prep=prepare(RAW,OUT)
assert prep['n']==32 and prep['groups']==4,prep
print(json.dumps(prep,indent=2))
shutil.copy2(ROOT/'pilot_annotators'/'annotator_A_FILLED.csv', A)
shutil.copy2(ROOT/'pilot_annotators'/'annotator_B_FILLED.csv', B)
report=adjudicate(FROZEN,A,B,ADJ,OUT)
print('Pilot annotation agreement:',report['decision_agreement'])
GOLD=OUT/'adjudicated_gold_PRIVATE.csv'
gold=pd.read_csv(GOLD)
print('Gold decision counts (PROVISIONAL):',gold.gold_decision.value_counts().to_dict())
assert len(gold)==32
print('Do not present these as final approved gold labels.')

## 3. Obtain and install the **exact current branch commit** of the real NiyamTrace-X source

In [ ]:
import subprocess, os, sys, secrets
SHA='d3df6130ab7615e3f3692828b8193e5ad62b8af7'
REPO_URL='https://github.com/bnssaanirudh/NiyamTrace-X.git'
REPO=Path('/content/niyamtrace_repo')
if not REPO.exists():
    subprocess.run(['git','clone','--depth','1','--branch','paper-closure/native-security-and-benchmarks',REPO_URL,str(REPO)],check=True)
actual=subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip()
if actual!=SHA:
    # The branch may have advanced since this notebook was created.
    subprocess.run(['git','fetch','--depth','1','origin',SHA],cwd=REPO,check=True)
    subprocess.run(['git','checkout','--detach',SHA],cwd=REPO,check=True)
actual=subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip()
assert actual==SHA,(actual,SHA)

# Offline parser -- no paid key, no real system access, no real ERP data.
os.environ['ENVIRONMENT']='test'
os.environ['LLM_BACKEND']='mock'
os.environ['NTX17_REPO_SOURCE']=str(REPO)
os.environ['NTX17_PILOT_WORK']=str(ROOT/'outputs'/'native_pilot')
os.environ['NIYAMTRACE_GATE_SECRET']=secrets.token_urlsafe(48)

# NOTE: this downloads real project dependencies; Colab internet is required.
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO/'niyamtrace')],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','scipy>=1.11,<2','pytest>=8,<10'],check=True)
print('Pinned source installed:',actual)
print('Run mode: mock/offline deterministic compiler + actual policy-gated pipeline')

## 4. Execute 32 × 2 paired **source-code** pilot arms (no fabricated results)

In [ ]:
import pandas as pd
RUN_NATIVE_PILOT=True
RUNNER=ROOT/'ntx_real_runner.py'
assert RUNNER.is_file(),RUNNER
if RUN_NATIVE_PILOT:
    # Run 32 compiler-only proposals and 32 full-source gate evaluations.
    # No errors are silently converted to BLOCK.
    from pilot_native_export import run_pilot
    pred_path=run_pilot(FROZEN,RUNNER,PRED,OUT/'native_pilot_diagnostics')
    pred=pd.read_csv(pred_path)
    assert len(pred)==64 and pred.case_id.nunique()==32
    print('Paired predictions:',len(pred))
    print(pred.groupby(['variant','decision']).size())
    assert pred.effect_audited.astype(str).str.lower().eq('false').all()
else:
    print('Source-pipeline experiment not executed.')

## 5. Pilot **verdict-only** diagnostic analysis (not publication evaluation)

In [ ]:
if PRED.is_file():
    PILOT=OUT/'pilot_source_offline_diagnostic'
    summary=evaluate(GOLD,PRED,PILOT,require_full_effects=False,
        min_cases=32,min_test_cases=8)
    assert summary['n_natural_total']==32
    print('PILOT ONLY — not manuscript evidence')
    print(json.dumps({k:summary[k] for k in
        ['status','n_natural_total','n_heldout_test','cohorts','paired_tests','per_language']},indent=2))
    print('Interpretation: Offline compiler proposals vs real policy gate, not LLM comparison.')
else:
    print('No predictions: run previous source-native cell; no metrics available.')

## 6. Export the **private pilot evidence** safely

In [ ]:
from google.colab import files
import zipfile,hashlib
if PRED.exists() and (OUT/'pilot_source_offline_diagnostic'/'experimental_results.json').is_file():
    archive=Path('/content/NTX17_Source_Offline_32_Pilot_Outputs_PRIVATE.zip')
    with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED) as z:
        for p in sorted(OUT.rglob('*')):
            if p.is_file():z.write(p,p.relative_to(ROOT))
        z.write(PRED,PRED.relative_to(ROOT))
    print('Private artifact:',archive,'bytes:',archive.stat().st_size)
    print('SHA256:',hashlib.sha256(archive.read_bytes()).hexdigest())
    print('Contains private de-identified user text and traces; do not publicly upload without permission.')
    files.download(str(archive))
else:
    print('No complete pilot yet — no output archive made.')

## 7. Research limitations and next actions
The 32-case annotation agreement is not a substitute for independent policy approval. The frozen labels contain no ALLOW cases and only four unique participants. This evaluation uses **mock/offline extraction**, not actual Granite or Qwen model generations. The compiler-only arm has no native tool execution, so it cannot support unsafe-effect comparisons. The genuine *NiyamTrace-X* arm uses a fresh synthetic database per case, but `effect_audited=false` remains until independently checking authorized state scopes against approved labels. **Do not copy the pilot accuracy into Scientific Reports or Frontiers as final independent validation.**